In [ ]:
import duckdb

import glob
from pathlib import Path

from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import timeit
import statistics

###################################################################
#     OPEN DB CONNECTION                                          #
###################################################################    

db = "Spectral_data.duckdb" 
con = duckdb.connect(db)

In [ ]:
###################################################################
#	 Create DB						                              #
###################################################################	

def drop_create_db(con):
	con.execute("""
	DROP TABLE IF EXISTS spectral; 
	DROP TABLE IF EXISTS sensor_info; 
	""")	

	con.execute("""
		CREATE TABLE sensor_info(
		"sensor" VARCHAR,
		"wavelenghts" DOUBLE[], 
		);
		""")

	con.execute("""
		CREATE TABLE spectral(
		"timestamp" TIMESTAMP,
		"sensor" VARCHAR,
		"ape" DOUBLE, 
		"irradiance" DOUBLE, 
		"spectra" DOUBLE[]		
		);
		""")



def insert_into_db(con, data):
	df = pd.DataFrame(data)
	df['timestamp'] = pd.to_datetime(df['timestamp'])

	column_order = ['timestamp', 'spectra', 'sensor']

	df = df.filter(column_order)

	con.execute("INSERT INTO spectral BY NAME SELECT timestamp, spectra, sensor FROM df")

def parse_file(filepath):
	"""
	Parse a spectral measurement CSV file.
	"""
	filepath = Path(filepath)
	with open(filepath, 'r') as f:
		file = f.readlines()
		meta_data = file[:7] 
		spectral_data = file[8:] 

	parsed_data = list()
	for idx_l, line in enumerate(meta_data):
		key = line.split(',')[0]
		# Forat each measurement
		for idx, data in enumerate(line.split(',')[1:-1]):
			match key:
				case 'Date':
					if data:
						data = datetime.strptime(data, '%Y/%m/%d')
				case 'Time':
					if data:
						data = datetime.strptime(data, '%H:%M:%S')
						data = timedelta(hours=data.hour, minutes=data.minute, seconds=data.second)
				case 'Exposure Time(msec)' | 'Sensor Temp.(deg)' | 'Power(V)':
					data = float(data) if data else float('nan')
				case 'Sensor':
					key = 'sensor'
				case _:
					pass
			
			# Use previous values if missing:  
			if key == 'Date' or key == 'Time' or key == 'Memo':
				miss_idx = idx-1
				while not data:
					data = parsed_data[miss_idx][key]
					miss_idx -= 1
				
			# Insert data 
			if idx_l == 0:
				parsed_data.append({key:data})
			else:
				parsed_data[idx].update({key:data})

	spectral_data = np.genfromtxt(spectral_data, delimiter=',', dtype=np.float64).T

	# Create datetime timestamp for each point and add spectral data
	for idx, datapoint in enumerate(parsed_data):
		data = parsed_data[idx]['Date'] + parsed_data[idx]['Time']
		parsed_data[idx].update({'timestamp':data})
		parsed_data[idx].update({'spectra':spectral_data[idx+1].tolist()})

	# Create a list of wavelengths
	wavelenghts = spectral_data[0].tolist()

	return parsed_data, wavelenghts


print("\n\n************************")
print("Connect to db:")

drop_create_db(con)

folder1 = "\\\\basisit.de\\share\\Outdoor-Data\\ZPV\\spectral\\WISER_35\\WSDac\\2025\\*.csv"
folder2 = "\\\\basisit.de\\share\\Outdoor-Data\\ZPV\\spectral\\WISER_35\\WSDac\\2026\\*.csv"

files = []
for folder in [folder1, folder2]:
	files.extend(glob.glob(folder))

# Parse files
timing = []
list_len = len(files)
for idx, filepath in enumerate(files):
	print(f"""{idx+1:04d}/{list_len:04d} --> {filepath}""")
	t1 = timeit.default_timer()

	try:
		parsed_data, wl = parse_file(filepath)
		insert_into_db(con, parsed_data)
	except (ValueError, FileNotFoundError, PermissionError) as e:
		print(f"""          ----> Error: : {type(e).__name__}: {e}""")
		continue  

	t2 = timeit.default_timer()
	timing.append(t2-t1)

print(f"""Mean file processing time: {statistics.fmean(timing)*1000:.3f} ms""")
print("************************")

con.sql("select * from spectral LIMIT 10;").show()

# For 2025+2026 via network:
# Mean file processing time: 132.701 ms

Sensor info

In [ ]:
# ALl sensors wl: 300-->1800
# MS-711, MS-712, Merge

data = list()
data.append({"sensor":"MS-711", "wavelenghts":np.arange(300,1800,1)})
data.append({"sensor":"MS-712", "wavelenghts":np.arange(300,1800,1)})
data.append({"sensor":"Merge",  "wavelenghts":np.arange(300,1800,1)})

df = pd.DataFrame(data)
column_order = ['sensor', 'wavelenghts']

df = df.filter(column_order)
con.execute("INSERT INTO sensor_info BY NAME SELECT sensor, wavelenghts FROM df")

con.sql("select * from sensor_info;").show()

In [30]:
con.close()